# modular-vlm-finetune — Kaggle runner

Trains one bridge module on top of frozen Vintern-1B on the full AutoViVQA data.
There is **no implicit sample cap** — pass `--limit` / `--smoke` yourself for short runs.

Sessions cap at ~12h; checkpoints go to `/kaggle/working/checkpoints/` and `--resume` picks up where you left off.


In [ ]:
BRANCH = 'chore/repo-restructure'   # change after merge to main
REPO   = 'https://github.com/nguyennn263/modular-vlm-finetune.git'

In [ ]:
import os
if not os.path.isdir('/kaggle/working/modular-vlm-finetune'):
    !git clone $REPO /kaggle/working/modular-vlm-finetune
%cd /kaggle/working/modular-vlm-finetune
!git fetch origin && git checkout $BRANCH && git pull

In [ ]:
!bash setup_kaggle.sh

### Data

The `nguynrichard/auto-vqabest` dataset is attached in the kernel metadata and mounted at
`/kaggle/input/`. `src/data/environment.py` resolves the path automatically — nothing to do.


In [ ]:
# sanity: resolve config + data paths without loading the model
!python -m src.cli.train --bridge residual --dry-run

In [ ]:
# SMOKE run — proves the pipeline trains + evaluates + checkpoints end to end.
# Swap for the full run below once this is green.
!python -m src.cli.train --bridge residual --smoke --output-dir /kaggle/working/checkpoints

In [ ]:
!python -m src.cli.evaluate --bridge residual \
    --checkpoint /kaggle/working/checkpoints/residual/best_model.pt --split val --limit 200

### P0 + P2 — grouped split (build then smoke-train on it)


In [ ]:
!python scripts/phase0_build_data.py

In [ ]:
!python -m src.cli.train --bridge residual --split-dir data/splits --smoke \
    --output-dir /kaggle/working/ckpt_split

### P1 — is n_tiles a real compute lever? (final-plan section 5.2)


In [ ]:
!python -m src.cli.profile --n-tiles 1 2 4 6 --samples 32

In [ ]:
import json, pathlib
p = pathlib.Path('outputs/profile/pipeline_cost.json')
print(json.loads(p.read_text()) if p.exists() else 'profile did not produce output')

### FULL training run

`--bridge {residual,multi_token,tile_attention,mini_qformer,qformer,gated_fusion}`


In [ ]:
# whole dataset, ~hours. Uncomment when ready.
# !python -m src.cli.train --bridge residual --output-dir /kaggle/working/checkpoints
# resume in a later session (auto-picks newest step_*.pt):
# !python -m src.cli.train --bridge residual --output-dir /kaggle/working/checkpoints --resume